<a href="https://colab.research.google.com/github/serdararici/breast-cancer-knn-svm-classification/blob/main/Breast_Cancer_KNN_vs_SVM_HSD_Bootcamp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🩺 Meme Kanseri Teşhisinde Makine Öğrenmesi: KNN vs SVM Karşılaştırması

**Huawei Student Developers – Veri Bilimi ve Makine Öğrenmesi Bootcamp Final Projesi**

---

**Hazırlayan:** Serdar Arıcı
**Tarih:** Ağustos 2026
**Bootcamp:** Türkiye Yapay Zeka Akademisi × Huawei Student Developers — Veri Bilimi ve Makine Öğrenmesi Bootcamp

---

## 📌 Proje Amacı

Bu projede, **Breast Cancer Wisconsin (Diagnostic) Dataset** kullanılarak, hücre çekirdeği ölçümlerinden yola çıkarak bir tümörün **iyi huylu (benign)** mu yoksa **kötü huylu (malignant)** mu olduğunu tahmin eden bir sınıflandırma modeli geliştirilecektir.

İki farklı makine öğrenmesi algoritması — **K-En Yakın Komşu (KNN)** ve **Destek Vektör Makineleri (SVM)** — eğitilecek, hiperparametreleri optimize edilecek ve performansları çeşitli metriklerle karşılaştırılacaktır.

## 📊 Veri Seti Hakkında

- **Kaynak:** [Kaggle - Breast Cancer Wisconsin (Diagnostic) Data Set](https://www.kaggle.com/datasets/uciml/breast-cancer-wisconsin-data)
- **Gözlem sayısı:** 569
- **Özellik sayısı:** 30 sayısal özellik (hücre çekirdeğinin yarıçapı, dokusu, çevresi, alanı, pürüzsüzlüğü vb. ölçümlerin ortalama, standart hata ve "en kötü" değerleri)
- **Hedef değişken:** `diagnosis` — M (Malignant / Kötü Huylu) veya B (Benign / İyi Huylu)

## 🛠️ İzlenecek Yol

1. Veri Yükleme ve Genel Bakış
2. Keşifsel Veri Analizi (EDA)
3. Veri Ön İşleme
4. Model 1: K-En Yakın Komşu (KNN)
5. Model 2: Destek Vektör Makineleri (SVM)
6. Model Karşılaştırması ve Değerlendirme
7. Sonuç ve Çıkarımlar

---

## 🔧 Gerekli Kütüphanelerin Yüklenmesi

Analiz, modelleme ve görselleştirme için kullanacağımız kütüphaneleri içe aktarıyoruz.

In [1]:
# Data manipulation libraries
import numpy as np
import pandas as pd

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn: preprocessing, models, and evaluation metrics
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, auc, ConfusionMatrixDisplay
)

# Set plotting style for a clean and professional look
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 100
plt.rcParams["font.size"] = 11

# Fix random seed for reproducibility across the whole notebook
RANDOM_STATE = 42

print("All libraries imported successfully.")

All libraries imported successfully.


## 1️⃣ Veri Yükleme ve Genel Bakış

Veri setini Google Drive üzerinden Colab ortamına bağlayarak okuyoruz. Veri seti, Kaggle'daki "Breast Cancer Wisconsin (Diagnostic) Data Set" kaynağından indirilmiştir.

In [6]:
# Mount Google Drive to access the dataset file
from google.colab import drive
drive.mount('/content/drive')

# Define the path to the dataset on Google Drive
DATA_PATH = "/content/drive/MyDrive/MachineLearning-AI-DataScience/TurkiyeYapayZekaAkademisi/finalProject/data.csv"

# Load the dataset into a pandas DataFrame
df = pd.read_csv(DATA_PATH)

print(f"Dataset shape: {df.shape}")
df.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset shape: (569, 33)


,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


Veri setinde **569 gözlem** ve **569 hasta kaydı, 30 sayısal özellik** bulunuyor. `id` sütunu sadece hasta kimlik numarası (modelleme için anlamsız), `Unnamed: 32` sütunu ise tamamen boş görünüyor — bu iki sütunu veri temizliği aşamasında çıkaracağız.

Her özellik, hücre çekirdeğinin bir karakteristiğini (yarıçap, doku, çevre, alan, pürüzsüzlük vb.) üç farklı istatistik üzerinden ifade ediyor: ortalama (`mean`), standart hata (`se`), ve en kötü değer (`worst`).

In [7]:
# General information about the dataset: column types and non-null counts
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 33 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   id                       569 non-null    int64  
 1   diagnosis                569 non-null    object 
 2   radius_mean              569 non-null    float64
 3   texture_mean             569 non-null    float64
 4   perimeter_mean           569 non-null    float64
 5   area_mean                569 non-null    float64
 6   smoothness_mean          569 non-null    float64
 7   compactness_mean         569 non-null    float64
 8   concavity_mean           569 non-null    float64
 9   concave points_mean      569 non-null    float64
 10  symmetry_mean            569 non-null    float64
 11  fractal_dimension_mean   569 non-null    float64
 12  radius_se                569 non-null    float64
 13  texture_se               569 non-null    float64
 14  perimeter_se             5

In [8]:
# Drop irrelevant columns:
# - 'id' is just a patient identifier, not useful for prediction
# - 'Unnamed: 32' is a completely empty column (artifact of the CSV export)
df = df.drop(columns=["id", "Unnamed: 32"])

print(f"Dataset shape after cleanup: {df.shape}")
df.head()

Dataset shape after cleanup: (569, 31)


,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678
